In [ ]:
import os

Globalne ustawienia

Struktura folderu
- labs2
    - case3_single.fastq.gz
    - copy_files
    - FastQC_reports
    - reports_case3
        - initial_report
        - trimming

In [ ]:
# ustawienie agenta ssh
agent = !echo $SSH_AUTH_SOCK
os.environ["SSH_AUTH_SOCK"] = agent[0] # to trzeba wydobyć
sv = 'ssh lamina'

# ustawienia fastp
js_set = '--disable_json'

# ustawienia ścieżek
fastq_file = 'labs2/case3_single.fastq.gz' # adres pliku
write_path = 'labs2/reports_case3' # adres folderu z raportami
copy_path = 'raporty' # adres lokalnego folderu dla plików 

In [ ]:
# wstępna analiza
loc_write_path = write_path + '/initial_report'
!{sv} "fastqc {fastq_file} -o {loc_write_path}"

Sprawdzamy czy są podane sekwencje UMI. Korzystamy ze znajomości formatu plików i będziemy przechodzić co 4 linie, żeby uzyskać informacje z samych nagłówków:

Struktura:
1. Nagłówek  - sekwencja UMI znajduje się na końcu po "UMI:
2. Sekwencja
3. Separator 
4. Jakości

In [ ]:
!{sv} "zcat  {fastq_file} | sed -n '1~4p' | grep -o "UMI:[A-Z]*""

Nie otrzymujemy żadnych wyników, zatem UMI nie były wykorzystane 

Chciałem sprawdzić czy wyniki z trimmingu będą się różnić, jeżeli
- wykonamy go automatycznie
- wprowadzimy ręcznie prefiks znaleziony w dokumentacji
W tym przypadku, oba są analogiczne, jednak będziemy korzystać z ręcznego trimmingu

W tym przypadku są takie same, zostawiam skrypt tylko na manualne - mamy pewność że nie usunie istotnych biologicznie sekwencji

In [ ]:
## ścieżki
loc_file_name = write_path + '/trimming_manual'
loc_write_path = write_path + '/trimming'
## informacyjnie jest to sekwencja która najczęściej występowała
# adapter_detected_sequence = "GATCGGAAGAGCACACGTCTGAACTCCAGTCACAGTCAACAATCTCGTATGCCGTCTTCTGCTTG"
## jest to prefix adapterów użytych w bibliotece 
adapter_universal_prefix = "GATCGGAAGAGCACACGTCTGAACTCCAGTCAC"

# usuwamy wszystkie adaptery z danym prefiksem 
!{sv} "fastp -i '{fastq_file}' \
    -o '{loc_file_name}.fastq.gz' --html '{loc_file_name}.html' --json '{loc_file_name}.json'\
    --adapter_sequence '{adapter_universal_prefix}'" 

# powtórzenie raportu
!{sv} "fastqc {loc_file_name}.fastq.gz -o {loc_write_path}" 


In [ ]:
# Ściągnięcie danych
!scp -r lamina:'{write_path}/*' {copy_path} 